# FableForge-14B — Free Colab Training with Unsloth

Train the FableForge-14B agent model on **Google Colab T4 (free tier)** using **Unsloth** for 2-5x faster training with 70% less VRAM.

## Pipeline Overview

| Stage | Purpose | Epochs | Time (T4) |
|-------|---------|--------|------------|
| 1 | Behavior Shaping (tool-use SFT) | 1 | ~6h |
| 2 | Skill Distillation (code SFT) | 1 | ~5h |
| 3 | Error Recovery (debug SFT) | 1 | ~2h |
| 4 | DPO Alignment (preference) | 1 | ~4h |

**Base model:** `Qwen/Qwen2.5-Coder-14B` (fits on T4 with Unsloth 4-bit quantization)

**Total training time on Colab T4:** ~17 hours (spread across sessions)

> **Tip:** Save checkpoints to Google Drive after each stage so you can resume if Colab disconnects.

In [ ]:
#@title Cell 1: Install Unsloth and Dependencies { display-mode: "form" }
#@markdown Installs Unsloth (2-5x faster, 70% less VRAM), transformers, PEFT, TRL, and supporting libs.
#@markdown This cell takes ~3-5 minutes on first run.

import os
import subprocess
import sys

# Check GPU
gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip()
print(f"GPU: {gpu_info}")

gpu_name = gpu_info.split(",")[0].strip()
vram_mb = int(gpu_info.split(",")[1].strip().replace("MiB", "").strip())
print(f"VRAM: {vram_mb}MB")

if vram_mb < 14000:
    print(f"⚠ WARNING: Only {vram_mb}MB VRAM. T4 (15GB) or better recommended.")
    print("  Consider using a smaller base model (Qwen2.5-Coder-7B) or reducing batch size.")

# Install Unsloth — uses prebuilt wheels for fast install
print("Installing Unsloth...")
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "unsloth[colab-new]",
                "--quiet"], check=True)

# Install core dependencies
print("Installing core dependencies...")
deps = [
    "transformers>=4.46.0",
    "datasets>=3.0.0",
    "accelerate>=1.1.0",
    "peft>=0.13.0",
    "trl>=0.12.0",
    "bitsandbytes>=0.43.0",
    "scipy",
    "sentencepiece",
    "protobuf",
    "huggingface_hub",
    "wandb",
]
subprocess.run([sys.executable, "-m", "pip", "install"] + deps + ["--quiet"], check=True)

# Verify Unsloth
try:
    import unsloth
    print(f"✓ Unsloth {unsloth.__version__} installed")
except ImportError:
    # Fallback: install from source
    subprocess.run([sys.executable, "-m", "pip", "install", "git+https://github.com/unslothai/unsloth.git",
                    "--quiet"], check=True)
    import unsloth
    print(f"✓ Unsloth {unsloth.__version__} installed (from source)")

import torch

print(f"✓ PyTorch {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"✓ CUDA memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
#@title Cell 2: Mount Google Drive for Persisting Checkpoints { display-mode: "form" }
#@markdown Mount Google Drive to save checkpoints between sessions.
#@markdown Checkpoints are saved to `/content/drive/MyDrive/fableforge-14b/`

from google.colab import drive

DRIVE_DIR = "/content/drive/MyDrive/fableforge-14b"

drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/data", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/exports", exist_ok=True)

print("✓ Google Drive mounted")
print(f"✓ Checkpoint directory: {DRIVE_DIR}/checkpoints")
print(f"  Data directory: {DRIVE_DIR}/data")
print(f"  Export directory: {DRIVE_DIR}/exports")

In [ ]:
#@title Cell 3: Download Fable-5 Dataset from HuggingFace { display-mode: "form" }
#@markdown Downloads the Fable-5 dataset (agent trajectories with tool-use, coding, error-recovery, and preference data).

import json
import os

from datasets import load_dataset

DATA_DIR = f"{DRIVE_DIR}/data"

# Download Fable-5 from HuggingFace (public dataset)
print("Downloading Fable-5 dataset from HuggingFace...")
ds = load_dataset("fableforge/fable-5", revision="main")

# Extract and stage-specific splits
splits_needed = {
    "behavior_shaping_train": None,
    "behavior_shaping_val": None,
    "skill_distillation_train": None,
    "skill_distillation_val": None,
    "error_recovery_train": None,
    "error_recovery_val": None,
    "dpo_train": None,
    "dpo_val": None,
}

for split_name in splits_needed:
    try:
        split_ds = ds[split_name]
        output_path = os.path.join(DATA_DIR, f"{split_name}.jsonl")
        with open(output_path, "w") as f:
            for example in split_ds:
                f.write(json.dumps(example) + "\n")
        splits_needed[split_name] = len(split_ds)
        print(f"  ✓ {split_name}: {len(split_ds)} examples → {output_path}")
    except KeyError:
        # If structured splits don't exist, we'll convert from the main split below
        print(f"  ℹ Split '{split_name}' not found, will convert from raw data")

# If structured splits don't exist, convert from the 'train' split
# using the trajectory-distiller
missing_splits = [k for k, v in splits_needed.items() if v is None]
if missing_splits:
    print(f"\nConverting {len(missing_splits)} splits from raw data...")
    import subprocess
    sys_path = "/tmp/fableforge/trajectory-distiller/src"
    if os.path.exists(sys_path):
        subprocess.run([
            sys.executable, "-m", "trajectory_distiller",
            "convert", "--dataset", "fableforge/fable-5",
            "--output-dir", DATA_DIR,
            "--stages", "behavior_shaping,skill_distillation,error_recovery,dpo",
        ], check=True)
    else:
        # Fallback: convert using built-in converter
        print("Using built-in data converter...")
        raw_ds = ds.get("train", ds.get("raw", None))
        if raw_ds is not None:
            import random
            random.seed(42)
            all_examples = list(raw_ds)
            random.shuffle(all_examples)
            val_count = max(1, int(len(all_examples) * 0.05))
            train_examples = all_examples[val_count:]
            val_examples = all_examples[:val_count]

            def format_behavior_shaping(example):
                """Convert raw trajectory to behavior shaping format."""
                messages = []
                if "conversations" in example:
                    for turn in example["conversations"]:
                        role = turn.get("role", turn.get("from", "user"))
                        content = turn.get("content", turn.get("value", ""))
                        if role == "human":
                            role = "user"
                        elif role == "assistant":
                            role = "assistant"
                        elif role == "system":
                            role = "system"
                        messages.append({"role": role, "content": content})
                elif "prompt" in example and "response" in example:
                    messages = [
                        {"role": "user", "content": example["prompt"]},
                        {"role": "assistant", "content": example["response"]},
                    ]
                else:
                    messages = [{"role": "user", "content": str(example)}]
                return {"messages": messages}

            for split_name in missing_splits:
                stage = split_name.replace("_train", "").replace("_val", "")
                is_val = "_val" in split_name
                data = val_examples if is_val else train_examples
                output_path = os.path.join(DATA_DIR, f"{split_name}.jsonl")
                with open(output_path, "w") as f:
                    for ex in data:
                        formatted = format_behavior_shaping(ex)
                        f.write(json.dumps(formatted) + "\n")
                print(f"  ✓ {split_name}: {len(data)} examples → {output_path}")

print(f"\n✓ Data ready in {DATA_DIR}")
print(f"  Total files: {len([f for f in os.listdir(DATA_DIR) if f.endswith('.jsonl')])}")

In [ ]:
#@title Cell 4: Convert Data Using Trajectory Distiller { display-mode: "form" }
#@markdown Converts raw Fable-5 trajectories into stage-specific training formats.
#@markdown Each stage needs a different data format for optimal learning.

import json
import os
import random

DATA_DIR = f"{DRIVE_DIR}/data"
random.seed(42)

def load_jsonl(path):
    """Load a JSONL file."""
    if not os.path.exists(path):
        return []
    data = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

def save_jsonl(data, path):
    """Save data as JSONL."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

# Load raw data
raw_data = load_jsonl(os.path.join(DATA_DIR, "behavior_shaping_train.jsonl"))
if not raw_data:
    raw_data = load_jsonl(os.path.join(DATA_DIR, "raw_train.jsonl"))

print(f"Loaded {len(raw_data)} raw examples")

# ──── Stage 1 Format: Behavior Shaping (tool-use conversations) ────
def to_behavior_shaping(example):
    """Format as multi-turn conversation with tool-use patterns."""
    if "messages" in example:
        return example
    prompt = example.get("prompt", example.get("instruction", ""))
    response = example.get("response", example.get("output", ""))
    system = (
        "You are FableForge, an expert AI coding agent. Use tools effectively. "
        "Always read files before editing. Test after changes. Provide clear explanations."
    )
    return {
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response},
        ]
    }

# ──── Stage 2 Format: Skill Distillation (code generation) ────
def to_skill_distillation(example):
    """Format as instruction → code pairs for coding excellence."""
    prompt = example.get("prompt", example.get("instruction", ""))
    response = example.get("response", example.get("output", ""))
    code_content = response if any(
        kw in response for kw in ["def ", "class ", "async ", "import ", "```python", "```js"]
    ) else ""
    if not code_content:
        input_text = example.get("input", "")
        code_content = response
    return {
        "instruction": f"Implement the following coding task:\n{prompt}",
        "input": example.get("input", ""),
        "output": response,
    }

# ──── Stage 3 Format: Error Recovery (error → fix pairs) ────
def to_error_recovery(example):
    """Format as error → recovery pairs for debugging expertise."""
    prompt = example.get("prompt", example.get("instruction", ""))
    response = example.get("response", example.get("output", ""))
    # Extract error context from the prompt/response
    error_type = "unknown"
    for et in ["SyntaxError", "TypeError", "NameError", "ValueError",
               "ImportError", "KeyError", "IndexError", "AttributeError",
               "RuntimeError", "OSError", "FileNotFoundError"]:
        if et.lower() in (prompt + response).lower():
            error_type = et
            break
    return {
        "instruction": f"Fix this {error_type} error:\n{prompt}",
        "input": "",
        "output": response,
        "error_type": error_type,
    }

# ──── Stage 4 Format: DPO (chosen/rejected pairs) ────
def to_dpo_pairs(example):
    """Convert to DPO format with chosen/rejected pairs."""
    prompt = example.get("prompt", example.get("instruction", ""))
    chosen = example.get("response", example.get("output", ""))
    # Generate a weaker rejected response (truncated, generic, or error-containing)
    lines = chosen.split("\n")
    rejected = "\n".join(lines[:max(1, len(lines) // 3)])  # Use first third
    if len(rejected) < 20:
        rejected = "I'll need more context to help with that."
    return {
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected,
    }

# Convert all stages
if raw_data:
    s1_data = [to_behavior_shaping(ex) for ex in raw_data]
    s2_data = [to_skill_distillation(ex) for ex in raw_data]
    s3_data = [to_error_recovery(ex) for ex in raw_data if
               any(kw in (ex.get("prompt", "") + ex.get("response", ""))
                   for kw in ["error", "Error", "fix", "debug", "traceback", "Traceback"])]
    s4_data = [to_dpo_pairs(ex) for ex in raw_data[:min(len(raw_data), 30000)]]

    # Split into train/val
    val_pct = 0.05

    for stage_name, stage_data in [("behavior_shaping", s1_data),
                                   ("skill_distillation", s2_data),
                                   ("error_recovery", s3_data),
                                   ("dpo", s4_data)]:
        random.shuffle(stage_data)
        val_count = max(1, int(len(stage_data) * val_pct))
        train_split = stage_data[val_count:]
        val_split = stage_data[:val_count]

        save_jsonl(train_split, os.path.join(DATA_DIR, f"{stage_name}_train.jsonl"))
        save_jsonl(val_split, os.path.join(DATA_DIR, f"{stage_name}_val.jsonl"))
        print(f"✓ {stage_name}: {len(train_split)} train, {len(val_split)} val")

print(f"\n✓ All stage data converted and saved to {DATA_DIR}")

In [ ]:
#@title Cell 5: Configure Unsloth QLoRA { display-mode: "form" }
#@markdown Loads the Qwen2.5-Coder-14B base model with Unsloth's optimized 4-bit quantization.
#@markdown **LoRA rank: 64, alpha: 128** — balances expressivity with VRAM efficiency on T4.

import torch
from unsloth import FastLanguageModel

# ──── Configuration ────
CONFIG = {
    "base_model": "Qwen/Qwen2.5-Coder-14B",
    # If 14B doesn't fit, fall back to 7B
    "fallback_model": "Qwen/Qwen2.5-Coder-7B",
    "max_seq_length": 4096,
    "load_in_4bit": True,
    "lora_r": 64,
    "lora_alpha": 128,
    "lora_dropout": 0.05,
    "lora_target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    "output_dir": f"{DRIVE_DIR}/checkpoints",
}

print("Loading model with Unsloth ...")
print(f"  Base model: {CONFIG['base_model']}")
print(f"  LoRA r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}")
print(f"  Max seq length: {CONFIG['max_seq_length']}")
print(f"  4-bit quantization: {CONFIG['load_in_4bit']}")

# Try loading 14B; fall back to 7B if OOM
base_model_name = CONFIG["base_model"]
model = None

for model_attempt in [CONFIG["base_model"], CONFIG["fallback_model"]]:
    try:
        print(f"\nAttempting to load: {model_attempt}")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_attempt,
            max_seq_length=CONFIG["max_seq_length"],
            load_in_4bit=CONFIG["load_in_4bit"],
            dtype=None,  # Auto-detect (bfloat16 on Ampere+, float16 on T4)
            trust_remote_code=True,
        )
        base_model_name = model_attempt
        print(f"✓ Loaded {model_attempt} successfully!")
        break
    except (torch.cuda.OutOfMemoryError, RuntimeError):
        print(f"✗ {model_attempt} OOM, falling back...")
        torch.cuda.empty_cache()
        continue

if model is None:
    raise RuntimeError("Could not load any model. Try restarting Colab runtime.")

# Print VRAM usage
vram_used = torch.cuda.memory_allocated() / 1e9
vram_reserved = torch.cuda.memory_reserved() / 1e9
print("\nVRAM usage after loading:")
print(f"  Allocated: {vram_used:.2f} GB")
print(f"  Reserved:  {vram_reserved:.2f} GB")

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"\n✓ Model ready: {base_model_name}")
print(f"  Vocab size: {len(tokenizer)}")
print(f"  Model dtype: {model.dtype}")

In [ ]:
#@title Cell 6: Stage 1 — Behavior Shaping (SFT on tool-use patterns) { display-mode: "form" }
#@markdown **Stage 1**: Fine-tune the model on Fable-5 tool-use trajectories.
#@markdown - 1 epoch, learning_rate 2e-4, LoRA r=64 alpha=128
#@markdown - Teaches the model when/how to use tools like Bash, Read, Edit
#@markdown - ~6 hours on T4

import os

import torch
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

STAGE = 1
STAGE_NAME = "behavior_shaping"
STAGE_OUTPUT = f"{CONFIG['output_dir']}/stage{STAGE}_{STAGE_NAME}"

print(f"{'='*60}")
print(f"  Stage {STAGE}: {STAGE_NAME}")
print(f"  LoRA r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}")
print("  LR=2e-4, 1 epoch, batch=2, grad_accum=8")
print(f"{'='*60}\n")

# Apply LoRA adapters to the base model
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized gradient checkpointing
    random_state=42,
    target_modules=CONFIG["lora_target_modules"],
)
model.print_trainable_parameters()

# Load stage 1 training data
train_path = os.path.join(DATA_DIR, f"{STAGE_NAME}_train.jsonl")
val_path = os.path.join(DATA_DIR, f"{STAGE_NAME}_val.jsonl")

train_ds = load_dataset("json", data_files=train_path, split="train")
val_ds = load_dataset("json", data_files=val_path, split="train") if os.path.exists(val_path) else None

print(f"Training examples: {len(train_ds)}")
if val_ds:
    print(f"Validation examples: {len(val_ds)}")

# Configure training for T4 (15GB VRAM)
training_args = SFTConfig(
    output_dir=STAGE_OUTPUT,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.06,
    bf16=False,  # T4 doesn't support bf16, use fp16
    fp16=True,
    logging_steps=10,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    eval_strategy="steps" if val_ds else "no",
    eval_steps=200 if val_ds else None,
    report_to="none",  # Disable wandb on Colab free tier
    max_seq_length=CONFIG["max_seq_length"],
    dataset_text_field="text",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

print("\nStarting Stage 1 training...")
trainer.train()

# Save adapter + tokenizer
trainer.save_model(os.path.join(STAGE_OUTPUT, "final"))
tokenizer.save_pretrained(os.path.join(STAGE_OUTPUT, "final"))
print(f"\n✓ Stage {STAGE} complete! Adapter saved to {STAGE_OUTPUT}/final")

# Also save to Google Drive
drive_checkpoint = f"{DRIVE_DIR}/checkpoints/stage{STAGE}_{STAGE_NAME}_final"
trainer.save_model(drive_checkpoint)
tokenizer.save_pretrained(drive_checkpoint)
print(f"✓ Checkpoint saved to Google Drive: {drive_checkpoint}")

In [ ]:
#@title Cell 7: Stage 2 — Skill Distillation (SFT on code generation) { display-mode: "form" }
#@markdown **Stage 2**: Fine-tune on high-quality code generation examples.
#@markdown - LoRA r=32, alpha=64 (reduced from stage 1 to preserve learned behavior)
#@markdown - Learning rate 1e-4, 1 epoch
#@markdown - ~5 hours on T4

import gc

import torch
from datasets import load_dataset
from peft import PeftModel
from trl import SFTConfig, SFTTrainer

STAGE = 2
STAGE_NAME = "skill_distillation"
STAGE_OUTPUT = f"{CONFIG['output_dir']}/stage{STAGE}_{STAGE_NAME}"

print(f"{'='*60}")
print(f"  Stage {STAGE}: {STAGE_NAME}")
print("  LoRA r=32, alpha=64 (reduced for skill refinement)")
print("  LR=1e-4, 1 epoch")
print(f"{'='*60}\n")

# Reload model fresh for stage 2 to avoid VRAM bloat
# Load base model + stage 1 adapter, then apply new LoRA
del model, trainer
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=CONFIG["load_in_4bit"],
    dtype=None,
    trust_remote_code=True,
)

# Merge stage 1 adapter into base model
stage1_adapter = f"{CONFIG['output_dir']}/stage1_behavior_shaping/final"
if not os.path.exists(stage1_adapter):
    stage1_adapter = f"{DRIVE_DIR}/checkpoints/stage1_behavior_shaping_final"

if os.path.exists(stage1_adapter):
    print(f"Loading stage 1 adapter: {stage1_adapter}")
    model = PeftModel.from_pretrained(model, stage1_adapter)
    model = model.merge_and_unload()
    print("✓ Stage 1 adapter merged into base model")
else:
    print("Warning: Stage 1 adapter not found. Training from base model.")

# Apply new LoRA adapters for stage 2 (smaller rank)
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    target_modules=CONFIG["lora_target_modules"],
)
model.print_trainable_parameters()

# Load stage 2 data
train_path = os.path.join(DATA_DIR, f"{STAGE_NAME}_train.jsonl")
val_path = os.path.join(DATA_DIR, f"{STAGE_NAME}_val.jsonl")

train_ds = load_dataset("json", data_files=train_path, split="train")
val_ds = load_dataset("json", data_files=val_path, split="train") if os.path.exists(val_path) else None

print(f"Training examples: {len(train_ds)}")

training_args = SFTConfig(
    output_dir=STAGE_OUTPUT,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.06,
    fp16=True,
    logging_steps=10,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    eval_strategy="steps" if val_ds else "no",
    eval_steps=200 if val_ds else None,
    report_to="none",
    max_seq_length=CONFIG["max_seq_length"],
    dataset_text_field="text",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(os.path.join(STAGE_OUTPUT, "final"))
tokenizer.save_pretrained(os.path.join(STAGE_OUTPUT, "final"))

# Save to Drive
drive_checkpoint = f"{DRIVE_DIR}/checkpoints/stage{STAGE}_{STAGE_NAME}_final"
trainer.save_model(drive_checkpoint)
tokenizer.save_pretrained(drive_checkpoint)
print(f"\n✓ Stage {STAGE} complete! Saved to {drive_checkpoint}")

In [ ]:
#@title Cell 8: Stage 3 — Error Recovery (SFT on debug/fix pairs) { display-mode: "form" }
#@markdown **Stage 3**: Fine-tune on error-recovery trajectories.
#@markdown - LoRA r=16, alpha=32 (narrower for specialized debugging)
#@markdown - Learning rate 5e-5, 1 epoch
#@markdown - ~2 hours on T4

import gc

import torch
from datasets import load_dataset
from peft import PeftModel
from trl import SFTConfig, SFTTrainer

STAGE = 3
STAGE_NAME = "error_recovery"
STAGE_OUTPUT = f"{CONFIG['output_dir']}/stage{STAGE}_{STAGE_NAME}"

print(f"{'='*60}")
print(f"  Stage {STAGE}: {STAGE_NAME}")
print("  LoRA r=16, alpha=32")
print("  LR=5e-5, 1 epoch")
print(f"{'='*60}\n")

# Reload model
try:
    del model, trainer
except:
    pass
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=CONFIG["load_in_4bit"],
    dtype=None,
    trust_remote_code=True,
)

# Merge stage 2 adapter
stage2_adapter = f"{CONFIG['output_dir']}/stage2_skill_distillation/final"
if not os.path.exists(stage2_adapter):
    stage2_adapter = f"{DRIVE_DIR}/checkpoints/stage2_skill_distillation_final"

if os.path.exists(stage2_adapter):
    print(f"Loading stage 2 adapter: {stage2_adapter}")
    model = PeftModel.from_pretrained(model, stage2_adapter)
    model = model.merge_and_unload()
    print("✓ Stage 2 adapter merged")
else:
    print("Warning: Stage 2 adapter not found. Training from base.")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model.print_trainable_parameters()

train_path = os.path.join(DATA_DIR, f"{STAGE_NAME}_train.jsonl")
val_path = os.path.join(DATA_DIR, f"{STAGE_NAME}_val.jsonl")

train_ds = load_dataset("json", data_files=train_path, split="train")
val_ds = load_dataset("json", data_files=val_path, split="train") if os.path.exists(val_path) else None
print(f"Training examples: {len(train_ds)}")

training_args = SFTConfig(
    output_dir=STAGE_OUTPUT,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.06,
    fp16=True,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    eval_strategy="steps" if val_ds else "no",
    eval_steps=100 if val_ds else None,
    report_to="none",
    max_seq_length=CONFIG["max_seq_length"],
    dataset_text_field="text",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    seed=42,
)

trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(os.path.join(STAGE_OUTPUT, "final"))
tokenizer.save_pretrained(os.path.join(STAGE_OUTPUT, "final"))

drive_checkpoint = f"{DRIVE_DIR}/checkpoints/stage{STAGE}_{STAGE_NAME}_final"
trainer.save_model(drive_checkpoint)
tokenizer.save_pretrained(drive_checkpoint)
print(f"\n✓ Stage {STAGE} complete! Saved to {drive_checkpoint}")

In [ ]:
#@title Cell 9: Stage 4 — DPO Alignment (preference optimization) { display-mode: "form" }
#@markdown **Stage 4**: Direct Preference Optimization for alignment.
#@markdown - Uses chosen/rejected pairs to align model with preferred outputs
#@markdown - LoRA r=16, alpha=32, DPO beta=0.1
#@markdown - Learning rate 5e-5, 1 epoch
#@markdown - ~4 hours on T4

import gc

import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel, TaskType
from trl import DPOConfig, DPOTrainer

STAGE = 4
STAGE_NAME = "dpo"
STAGE_OUTPUT = f"{CONFIG['output_dir']}/stage{STAGE}_{STAGE_NAME}"

print(f"{'='*60}")
print(f"  Stage {STAGE}: DPO Alignment")
print("  LoRA r=16, alpha=32, DPO beta=0.1")
print("  LR=5e-5, 1 epoch")
print(f"{'='*60}\n")

try:
    del model, trainer
except:
    pass
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=CONFIG["load_in_4bit"],
    dtype=None,
    trust_remote_code=True,
)

# Merge stage 3 adapter into base
stage3_adapter = f"{CONFIG['output_dir']}/stage3_error_recovery/final"
if not os.path.exists(stage3_adapter):
    stage3_adapter = f"{DRIVE_DIR}/checkpoints/stage3_error_recovery_final"

if os.path.exists(stage3_adapter):
    print(f"Loading stage 3 adapter: {stage3_adapter}")
    model = PeftModel.from_pretrained(model, stage3_adapter)
    model = model.merge_and_unload()
    print("✓ Stage 3 adapter merged")

# Apply new LoRA for DPO
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# Load reference model for DPO (same base, no adapter)
ref_model, _ = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=CONFIG["load_in_4bit"],
    dtype=None,
    trust_remote_code=True,
)

dpo_train_path = os.path.join(DATA_DIR, "dpo_train.jsonl")
dpo_ds = load_dataset("json", data_files=dpo_train_path, split="train")
print(f"DPO pairs: {len(dpo_ds)}")

training_args = DPOConfig(
    output_dir=STAGE_OUTPUT,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=10,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    report_to="none",
    max_length=4096,
    max_prompt_length=2048,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    beta=0.1,
    loss_type="sigmoid",
    seed=42,
)

trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=training_args,
    train_dataset=dpo_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.train()
trainer.save_model(os.path.join(STAGE_OUTPUT, "final"))
tokenizer.save_pretrained(os.path.join(STAGE_OUTPUT, "final"))

drive_checkpoint = f"{DRIVE_DIR}/checkpoints/stage{STAGE}_{STAGE_NAME}_final"
trainer.save_model(drive_checkpoint)
tokenizer.save_pretrained(drive_checkpoint)
print(f"\n✓ Stage {STAGE} (DPO) complete! Saved to {drive_checkpoint}")

In [ ]:
#@title Cell 10: Evaluate on BenchAgent Tasks { display-mode: "form" }
#@markdown Quick evaluation of the trained model on coding agent benchmarks.
#@markdown Tests tool-use, code generation, and error recovery capabilities.

import json
import os

print("=" * 60)
print("  Evaluation: BenchAgent Tasks")
print("=" * 60)

# Load the final trained model for inference
import torch
from unsloth import FastLanguageModel

final_adapter = f"{CONFIG['output_dir']}/stage4_dpo/final"
if not os.path.exists(final_adapter):
    final_adapter = f"{DRIVE_DIR}/checkpoints/stage4_dpo_final"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
    trust_remote_code=True,
)

# Load the DPO adapter for inference
if os.path.exists(final_adapter):
    from peft import PeftModel
    # Merge all adapters sequentially
    for stage_name in ["behavior_shaping", "skill_distillation", "error_recovery", "dpo"]:
        adapter_path = f"{DRIVE_DIR}/checkpoints/stage{['behavior_shaping','skill_distillation','error_recovery','dpo'].index(stage_name)+1}_{stage_name}_final"
        if os.path.exists(adapter_path):
            model = PeftModel.from_pretrained(model, adapter_path)
            model = model.merge_and_unload()
            print(f"✓ Merged adapter: {stage_name}")

FastLanguageModel.for_inference(model)

# Define evaluation prompts
eval_prompts = [
    {
        "task": "tool_use",
        "prompt": "Read the file src/main.py and fix the syntax error on line 42.",
        "expected_tools": ["Read", "Edit"],
    },
    {
        "task": "code_generation",
        "prompt": "Implement a function that finds the longest common subsequence of two strings.",
        "expected_tools": [],
    },
    {
        "task": "error_recovery",
        "prompt": "The following code raises a TypeError: 'NoneType' object is not subscriptable.\n\ndef get_user(id):\n    user = db.find(id)\n    return user['name']\n\nFix this bug.",
        "expected_tools": [],
    },
]

results = []
for eval_item in eval_prompts:
    prompt = eval_item["prompt"]
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    results.append({
        "task": eval_item["task"],
        "prompt": prompt[:100],
        "response": response[:500],
        "response_length": len(response),
    })
    print(f"\n--- Task: {eval_item['task']} ---")
    print(f"Prompt: {prompt[:100]}...")
    print(f"Response ({len(response)} chars): {response[:300]}...")

# Save eval results
eval_output = os.path.join(DRIVE_DIR, "evaluation_results.json")
with open(eval_output, "w") as f:
    json.dump(results, f, indent=2)
print(f"\n✓ Evaluation results saved to {eval_output}")
print(f"  Tasks evaluated: {len(results)}")

In [ ]:
#@title Cell 11: Export to GGUF and 16-bit Formats { display-mode: "form" }
#@markdown Export the final merged model to GGUF (for llama.cpp/ollama) and 16-bit (for vLLM/tgi).
#@markdown Saves to Google Drive for easy download.

import gc

import torch

EXPORT_DIR = f"{DRIVE_DIR}/exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

print("=" * 60)
print("  Export: GGUF + 16-bit")
print("=" * 60)

# Step 1: Merge all adapters into base model and save 16-bit
print("\n[1/3] Merging all adapters into base model...")

try:
    del model
except:
    pass
gc.collect()
torch.cuda.empty_cache()

from peft import PeftModel
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=False,  # Load in full precision for merging
    dtype=torch.bfloat16,
    trust_remote_code=True,
)

# Merge adapters sequentially
stages = [
    ("behavior_shaping", 1),
    ("skill_distillation", 2),
    ("error_recovery", 3),
    ("dpo", 4),
]
for stage_name, stage_num in stages:
    adapter_path = f"{DRIVE_DIR}/checkpoints/stage{stage_num}_{stage_name}_final"
    if os.path.exists(adapter_path):
        model = PeftModel.from_pretrained(model, adapter_path)
        model = model.merge_and_unload()
        print(f"  ✓ Merged stage {stage_num}: {stage_name}")
    else:
        print(f"  ⚠ Adapter not found: {adapter_path}")

# Save 16-bit merged model
merged_16bit_path = f"{EXPORT_DIR}/fableforge-14b-merged-16bit"
print(f"\n[2/3] Saving 16-bit model to {merged_16bit_path}...")
model.save_pretrained(merged_16bit_path)
tokenizer.save_pretrained(merged_16bit_path)
print("  ✓ 16-bit model saved")

# Step 2: Export to GGUF using Unsloth's built-in converter
print("\n[3/3] Exporting to GGUF...")
try:
    # Try Unsloth's GGUF export first
    model.save_pretrained_gguf(
        f"{EXPORT_DIR}/fableforge-14b-Q4_K_M",
        tokenizer,
        quantization_method="q4_k_m",
    )
    print(f"  ✓ GGUF (Q4_K_M) saved to {EXPORT_DIR}/fableforge-14b-Q4_K_M")
except Exception as e:
    print(f"  ⚠ Unsloth GGUF export failed: {e}")
    print("  Falling back to llama.cpp conversion...")
    # Fallback: use llama.cpp
    llama_cpp_dir = f"{EXPORT_DIR}/llama.cpp"
    if not os.path.exists(llama_cpp_dir):
        subprocess.run(["git", "clone", "https://github.com/ggerganov/llama.cpp.git",
                       llama_cpp_dir, "--depth", "1"], check=False)
    convert_script = os.path.join(llama_cpp_dir, "convert_hf_to_gguf.py")
    if os.path.exists(convert_script):
        subprocess.run([
            sys.executable, convert_script,
            merged_16bit_path,
            "--outfile", f"{EXPORT_DIR}/fableforge-14b-f16.gguf",
            "--outtype", "f16",
        ], check=False)
        print(f"  ✓ GGUF (f16) saved to {EXPORT_DIR}/fableforge-14b-f16.gguf")
    else:
        print("  ⚠ llama.cpp not available for GGUF export")
        print("     Install llama.cpp and run manually:")
        print(f"     python convert_hf_to_gguf.py {merged_16bit_path} --outfile fableforge-14b-Q4_K_M.gguf")

print("\n✓ Export complete!")

In [ ]:
#@title Cell 12: Upload to HuggingFace Hub { display-mode: "form" }
#@markdown Upload the final model to HuggingFace Hub for sharing.
#@markdown Set your HF token in the input field below.

import getpass

from google.colab import userdata
from huggingface_hub import HfApi, create_repo

HF_USERNAME = "fableforge" #@param {type:"string"}
MODEL_NAME = "FableForge-14B" #@param {type:"string"}
UPLOAD_16BIT = True #@param {type:"boolean"}
UPLOAD_GGUF = True #@param {type:"boolean"}

# Get HF token
try:
    hf_token = userdata.get("HF_TOKEN")
except:
    hf_token = getpass.getpass("Enter your HuggingFace token: ")

from huggingface_hub import login

login(token=hf_token)

api = HfApi()
repo_id = f"{HF_USERNAME}/{MODEL_NAME}"

# Create repo
try:
    create_repo(repo_id, exist_ok=True, token=hf_token)
    print(f"✓ Created repo: {repo_id}")
except Exception as e:
    print(f"Repo may already exist: {e}")

# Upload 16-bit model
if UPLOAD_16BIT:
    merged_path = f"{DRIVE_DIR}/exports/fableforge-14b-merged-16bit"
    if os.path.exists(merged_path):
        print(f"Uploading 16-bit model to {repo_id}...")
        api.upload_folder(
            folder_path=merged_path,
            repo_id=repo_id,
            commit_message="Upload FableForge-14B 16-bit merged model",
            token=hf_token,
        )
        print("  ✓ 16-bit model uploaded")
    else:
        print("  ⚠ 16-bit model not found, skipping")

# Upload GGUF
if UPLOAD_GGUF:
    gguf_path = f"{DRIVE_DIR}/exports/fableforge-14b-Q4_K_M.Q4_K_M.gguf"
    if not os.path.exists(gguf_path):
        # Check alternate location
        for f in os.listdir(f"{DRIVE_DIR}/exports"):
            if f.endswith(".gguf"):
                gguf_path = os.path.join(f"{DRIVE_DIR}/exports", f)
                break

    if os.path.exists(gguf_path):
        print(f"Uploading GGUF to {repo_id}...")
        api.upload_file(
            path_or_fileobj=gguf_path,
            path_in_repo=os.path.basename(gguf_path),
            repo_id=repo_id,
            commit_message="Upload FableForge-14B Q4_K_M GGUF",
            token=hf_token,
        )
        print("  ✓ GGUF uploaded")
    else:
        print("  ⚠ GGUF file not found, skipping")

# Upload model card
model_card = """---
language:
- en
library_name: peft
tags:
- code
- agent
- fableforge
base_model: Qwen/Qwen2.5-Coder-14B
---

# FableForge-14B

A 14B coding agent model fine-tuned from Qwen2.5-Coder-14B using the Fable-5 dataset.

## Training Pipeline

| Stage | Description | Method |
|-------|-------------|--------|
| 1 | Behavior Shaping | SFT on tool-use trajectories |
| 2 | Skill Distillation | SFT on code generation |
| 3 | Error Recovery | SFT on debug/fix pairs |
| 4 | DPO Alignment | Preference optimization |

Trained with **Unsloth** on Google Colab T4 (free tier).

## Usage

```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="fableforge/FableForge-14B",
    max_seq_length=4096,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

inputs = tokenizer("Implement a REST API endpoint", return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=512)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```
"""

api.upload_file(
    path_or_fileobj=model_card.encode(),
    path_in_repo="README.md",
    repo_id=repo_id,
    commit_message="Add model card",
    token=hf_token,
)
print("  ✓ Model card uploaded")

print(f"\n✓ Model published at: https://huggingface.co/{repo_id}")